# Physical AI Platform - Circuit Backend Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**License:** GNU General Public License v3.0-only (GPL-3.0-only). Third-party simulators, Python packages, PDKs and model libraries retain their own licenses.

This notebook is an optional, user-initiated Google Colab execution environment for the circuit package. It is not the canonical system model, not the platform kernel, and not a safety/signoff authority.

## What this notebook does

1. Accepts the separately delivered `circuit.zip` through the browser upload UI.
2. Verifies the package SHA-256 manifest.
3. Runs the browser-safe native solver tests and Node.js backend contract tests.
4. Installs ngspice in this ephemeral Colab Linux VM when permitted, then runs actual ngspice samples.
5. Attempts Xyce only if an approved executable is present; absence is SKIP, not PASS.
6. Exports an execution report with backend versions and PASS/FAIL/SKIP status.

**Browser-only platform rule:** GitHub Pages remains static/browser-only. Native simulators are never executed in the user's browser or on the user's local machine. Colab is an optional cloud runtime. Data uploaded here is processed by Google-hosted compute; do not upload confidential, export-controlled, proprietary PDK or otherwise restricted data without authorization.

**Open Google Colab:** [https://colab.research.google.com/](https://colab.research.google.com/). **Open-in-Colab repository deep link:** after Grok commits this notebook, the repository-specific deep link is documented in `COLAB.md`. The current supplied archive does not expose a repository owner/name/branch, so this notebook does not invent one.


## 0. Runtime disclosure and execution boundary

The native `module.js` is browser-compatible JavaScript. The external backend adapter uses Node.js process and filesystem APIs and must run only in a trusted compute environment such as this explicitly started Colab VM or a future controlled platform worker. Colab sessions are ephemeral. Download reports that need to be retained.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile, textwrap, time
from google.colab import files

WORK = Path('/content/pap-circuit-lab')
WORK.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Node present:', bool(shutil.which('node')))
print('ngspice present:', bool(shutil.which('ngspice')))
print('Xyce present:', bool(shutil.which('Xyce') or shutil.which('xyce')))

## 1. Upload the circuit package ZIP

Upload the `circuit.zip` handoff artifact. Do not upload unapproved third-party PDK/model files. The package archive itself contains the copyright, license, manifest and integrity files.

In [ ]:
uploaded = files.upload()
zip_candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
if not zip_candidates:
    raise RuntimeError('No ZIP uploaded. Upload circuit.zip.')
selected = next((p for p in zip_candidates if p.name == 'circuit.zip'), zip_candidates[0])
extract = WORK / 'extract'
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir(parents=True)
with zipfile.ZipFile(selected) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f'ZIP CRC failure: {bad}')
    for info in zf.infolist():
        name = info.filename.replace('\\', '/')
        parts = Path(name).parts
        if name.startswith('/') or '..' in parts or not parts:
            raise RuntimeError(f'Unsafe ZIP member path: {info.filename}')
        mode = (info.external_attr >> 16) & 0o170000
        if mode == 0o120000:
            raise RuntimeError(f'Symlink ZIP member rejected: {info.filename}')
        target = (extract / name).resolve()
        if extract.resolve() not in target.parents and target != extract.resolve():
            raise RuntimeError(f'ZIP member escapes extraction root: {info.filename}')
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)
package_dirs = list(extract.rglob('manifest.json'))
if not package_dirs:
    raise RuntimeError('manifest.json not found in uploaded archive')
PKG = package_dirs[0].parent
print('Package:', PKG)
print((PKG / 'manifest.json').read_text()[:1200])

## 2. Verify the SHA-256 package manifest

Every listed file must match. The manifest itself is not self-hashed; this is expected.

In [ ]:
manifest = PKG / 'MANIFEST.sha256'
if not manifest.exists():
    raise RuntimeError('MANIFEST.sha256 is missing')
checks = []
for line in manifest.read_text().splitlines():
    if not line.strip():
        continue
    expected, rel = line.split(None, 1)
    rel = rel.strip().lstrip('*')
    target = (PKG / rel).resolve()
    if PKG.resolve() not in target.parents:
        raise RuntimeError(f'Unsafe manifest path: {rel}')
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    checks.append((rel, expected == actual))
failed = [x for x in checks if not x[1]]
print(f'Integrity: {len(checks)-len(failed)}/{len(checks)} verified')
if failed:
    raise RuntimeError(f'Integrity failures: {failed}')

## 3. Install the optional execution tools in this Colab VM

This does not install anything on the user device or alter the GitHub Pages site. The install is ephemeral and subject to Colab image and package repository availability.

In [ ]:
def run_cmd(args, check=True):
    print('$', ' '.join(map(str, args)))
    return subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=check)

if not shutil.which('node'):
    run_cmd(['apt-get', 'update', '-qq'])
    run_cmd(['apt-get', 'install', '-y', 'nodejs', 'npm'])
if not shutil.which('ngspice'):
    run_cmd(['apt-get', 'update', '-qq'])
    install = run_cmd(['apt-get', 'install', '-y', 'ngspice'], check=False)
    print(install.stdout[-5000:])
print('Node:', run_cmd(['node', '--version'], check=False).stdout.strip())
if shutil.which('ngspice'):
    print('ngspice:', run_cmd(['ngspice', '-v'], check=False).stdout[-1200:])
else:
    print('ngspice unavailable: real ngspice tests will be SKIP, not PASS')

## 4. Run native solver and adapter contract tests

The adapter contract suite uses deterministic fake process runners. It validates generated decks, output parsing, process failures, model-pack checks and HOLD behavior. It does not count as a real ngspice/Xyce run.

In [ ]:
def run_test(args, cwd):
    p = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {args}')
    return p.stdout

run_test(['node', 'test.mjs'], PKG)
run_test(['node', 'backends/test.mjs'], PKG)

## 5. Run real-engine reference samples

This invokes the actual external executable. The harness compares the RC, RL and divider samples and the nonlinear diode-clamp sample against declared reference tolerances. An absent engine is recorded as SKIP. A present engine that fails or disagrees with a reference causes a non-zero exit.

In [ ]:
report_path = WORK / 'real-engine-report.json'
env = os.environ.copy()
env['CIRCUIT_REAL_ENGINE_REPORT'] = str(report_path)
p = subprocess.run(['node', 'backends/test-real-engines.mjs'], cwd=PKG, env=env,
                   text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
if report_path.exists():
    real_report = json.loads(report_path.read_text())
    print(json.dumps(real_report['totals'], indent=2), real_report['status'])
else:
    real_report = {'status': 'NOT_RUN', 'reason': 'real-engine report was not created'}
real_engine_exit_code = p.returncode
print('Real-engine harness exit code:', real_engine_exit_code)
print('Note: SKIP is not PASS. A FAIL report is still exported in the final cell.')

## 6. Optional Xyce installation

Xyce is not installed by a generic apt command here because supported package/build variants depend on platform and build configuration. Supply an approved, compatible Xyce executable to the Colab VM using the project's controlled artifact distribution and license process. Then rerun section 5. Do not download arbitrary binaries or model packs from untrusted sources.

In [ ]:
xyce_path = shutil.which('Xyce') or shutil.which('xyce')
print('Xyce executable:', xyce_path or 'NOT AVAILABLE (real-engine tests will SKIP)')
if xyce_path:
    print(run_cmd([xyce_path, '-v'], check=False).stdout[-2000:])

## 7. Export run report

The report records actual engine execution, skips and provenance. It does not promote platform fidelity or physical verification. Grok must bind a reviewed report to canonical component/configuration identities after stitching.

In [ ]:
run_manifest = {
    'package': 'physical-ai-platform/drop-in/circuit',
    'package_version': json.loads((PKG/'manifest.json').read_text())['packageVersion'],
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'node_version': run_cmd(['node','--version'], check=False).stdout.strip(),
    'ngspice_available': bool(shutil.which('ngspice')),
    'xyce_available': bool(shutil.which('Xyce') or shutil.which('xyce')),
    'real_engine_report': real_report,
    'evidence_state': 'UNVERIFIED',
    'physical_verification': 'NOT_PERFORMED',
    'fidelity_promotion': 'NOT_REQUESTED'
}
out = WORK / 'circuit-colab-run-manifest.json'
out.write_text(json.dumps(run_manifest, indent=2) + '\n')
print(out)
files.download(str(out))
if report_path.exists():
    files.download(str(report_path))
if real_engine_exit_code != 0:
    raise RuntimeError('One or more installed real-engine reference tests failed; report was exported above.')

## Copyright, attribution and license

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.

GNU General Public License v3.0-only (GPL-3.0-only). Third-party simulators, Python packages, PDKs and model libraries retain their own licenses.

No third-party simulator binary, PDK or foundry model is included in this notebook. ngspice, Xyce, Node.js, Google Colab and any separately supplied model libraries are independent works governed by their respective licenses and terms. The package's GPL license does not relicense external model data. Verify license and export/privacy authorization before uploading models or designs.

**Engineering disclaimer:** numerical simulation is not physical validation, safety certification, foundry signoff or authorization to manufacture. All results require engineering review and appropriate independent verification.